# 003 Tools And Tool Calling

这是 LangChain 学习线的第三份 Notebook。

官方参考：

- https://docs.langchain.com/oss/python/langchain/tools
- https://docs.langchain.com/oss/python/langchain/agents

学习目标：

1. 学会用 `@tool` 把 Python 函数声明成 LangChain tool
2. 看懂 tool 的 name、description、args schema
3. 理解模型看到的是工具 schema，不是 Python 函数本身
4. 对比 LangChain tool 和本仓库 `ToolDefinition`
5. 理解为什么真实系统还需要权限、审批和执行边界

---

## 1. Tool 的本质

Tool 不是“模型会执行 Python 函数”。

更准确地说：

```text
Python 函数
  -> 工具声明 schema
  -> 发给模型
  -> 模型选择工具名和参数
  -> 应用代码执行函数
  -> 工具结果再回到模型上下文
```

这和本仓库 `ToolDefinition` 的设计完全一致：模型只看到 `name / description / parameters`，真正的 `handler` 留在服务端。

In [ ]:
%pip install -U langchain langchain-openai python-dotenv

## 2. 定义第一个 LangChain Tool

`@tool` 会读取函数名、类型标注和 docstring，生成模型可理解的工具描述。

In [1]:
from langchain_core.tools import tool


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b


print("name:", add_numbers.name)
print("description:", add_numbers.description)
print("args:", add_numbers.args)
print("invoke:", add_numbers.invoke({"a": 2, "b": 3}))

name: add_numbers
description: Add two integers and return the sum.
args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}
invoke: 5


## 3. 查看 Tool Schema

模型真正看到的是 schema。

这一步很重要：tool calling 的稳定性，强依赖 schema 是否清晰。

In [2]:
import json

schema = add_numbers.args_schema.model_json_schema()
print(json.dumps(schema, ensure_ascii=False, indent=2))

{
  "description": "Add two integers and return the sum.",
  "properties": {
    "a": {
      "title": "A",
      "type": "integer"
    },
    "b": {
      "title": "B",
      "type": "integer"
    }
  },
  "required": [
    "a",
    "b"
  ],
  "title": "add_numbers",
  "type": "object"
}


## 4. 用 Pydantic 明确参数说明

只靠函数名和 docstring 有时不够。复杂参数建议使用 Pydantic schema，让字段说明更稳定。

In [3]:
from typing import Literal

from pydantic import BaseModel, Field


class WeatherArgs(BaseModel):
    location: str = Field(..., description="City name, for example Shanghai or Beijing.")
    unit: Literal["celsius", "fahrenheit"] = Field("celsius", description="Temperature unit.")


@tool(args_schema=WeatherArgs)
def get_demo_weather(location: str, unit: str = "celsius") -> str:
    """Get demo weather for a city. This is a deterministic teaching tool."""
    suffix = "C" if unit == "celsius" else "F"
    value = 25 if unit == "celsius" else 77
    return f"{location}: sunny, {value}°{suffix}"


print(json.dumps(get_demo_weather.args_schema.model_json_schema(), ensure_ascii=False, indent=2))
print(get_demo_weather.invoke({"location": "Shanghai"}))

{
  "properties": {
    "location": {
      "description": "City name, for example Shanghai or Beijing.",
      "title": "Location",
      "type": "string"
    },
    "unit": {
      "default": "celsius",
      "description": "Temperature unit.",
      "enum": [
        "celsius",
        "fahrenheit"
      ],
      "title": "Unit",
      "type": "string"
    }
  },
  "required": [
    "location"
  ],
  "title": "WeatherArgs",
  "type": "object"
}
Shanghai: sunny, 25°C


## 5. 和本仓库 ToolDefinition 对比

本仓库的 `ToolDefinition` 多了两个非常工程化的字段：

- `risk_level`
- `parallel_safe`

LangChain tool 更关注“如何让模型知道工具可用”。

本仓库 ToolDefinition 还关注：

- 是否允许自动执行
- 是否需要审批
- 是否可以并行
- handler 是否被系统托管

这就是框架抽象和 Harness 控制面的差异。

In [4]:
from app.agents.types import ToolDefinition


def wrap_langchain_tool_as_tool_definition(langchain_tool, risk_level: str = "low") -> ToolDefinition:
    return ToolDefinition(
        name=langchain_tool.name,
        description=langchain_tool.description,
        risk_level=risk_level,
        parallel_safe=True,
        handler=lambda arguments: {"output": langchain_tool.invoke(arguments)},
        input_schema=langchain_tool.args_schema.model_json_schema(),
    )


definition = wrap_langchain_tool_as_tool_definition(get_demo_weather)
print(definition.name)
print(definition.risk_level)
print(json.dumps(definition.input_schema, ensure_ascii=False, indent=2))

get_demo_weather
low
{
  "properties": {
    "location": {
      "description": "City name, for example Shanghai or Beijing.",
      "title": "Location",
      "type": "string"
    },
    "unit": {
      "default": "celsius",
      "description": "Temperature unit.",
      "enum": [
        "celsius",
        "fahrenheit"
      ],
      "title": "Unit",
      "type": "string"
    }
  },
  "required": [
    "location"
  ],
  "title": "WeatherArgs",
  "type": "object"
}


## 6. 可选：让 Agent 使用 Tool

如果你已经配置好 `.env`，可以让 LangChain agent 自己决定是否调用 `get_demo_weather`。

这一步会产生真实模型调用。没有 API key 时可以跳过。

In [5]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI


def load_project_env() -> Path | None:
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=False)
            return env_path
    return None


load_project_env()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

if not OPENAI_API_KEY:
    agent = None
    print("Skip live agent call because OPENAI_API_KEY is missing.")
else:
    model = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)
    agent = create_agent(
        model=model,
        tools=[get_demo_weather],
        system_prompt="你是一个中文教学助手。需要天气信息时使用工具。",
    )
    print("Agent ready")

Agent ready


In [6]:
if agent is None:
    print("Skip invoke.")
else:
    result = agent.invoke({
        "messages": [
            {"role": "user", "content": "上海今天的演示天气如何？"}
        ]
    })
    last = result["messages"][-1]
    print(last.content)



上海今天的演示天气是晴天，气温为 25°C。


## 7. 为什么还需要权限层

LangChain 可以让模型知道工具并调用工具，但真实业务里还要回答这些问题：

1. 这个工具是否低风险？
2. 是否允许自动执行？
3. 是否要用户审批？
4. 参数是否越权？
5. 工具失败后如何恢复？

这也是本仓库 `ToolRegistry.decide_permission(...)` 和 approval checkpoint 的意义。

所以不要把 tool calling 理解为“模型调用函数”。更准确地说：

```text
模型提出工具调用请求，系统决定是否执行。
```

## 8. 本讲小结

这一讲先记住四点：

1. `@tool` 把 Python 函数变成模型可见的工具 schema。
2. 模型看到的是 `name / description / args schema`，不是 Python handler。
3. LangChain tool 适合快速声明模型工具能力。
4. 真实工程仍然需要权限、审批、ledger 和恢复机制。

下一讲建议学习 structured output：

- 为什么不要只靠 prompt 要求模型返回 JSON
- 如何用 schema 固定模型输出
- 如何改造 planner action